# Ejercicio 4 — Agrupamiento de clientes según comportamiento de compra

**Asignatura:** Análisis de Datos  
**Carrera:** Ingeniería en Ciencia de Datos  
**Evento evaluativo:** 4  
**Equipo:** 7  
**Dataset:** Mall Customers Dataset

---

## 1. Descripción del problema

En el ámbito del marketing y la gestión comercial, comprender el comportamiento de los clientes es esencial para diseñar estrategias de segmentación efectivas. El **Mall Customers Dataset** contiene información demográfica y de consumo de 200 clientes de un centro comercial: género, edad, ingreso anual estimado y una puntuación de gasto asignada por el mall.

Al no contar con una variable objetivo predefinida, este es un problema de **aprendizaje no supervisado**: el objetivo es descubrir grupos naturales (clusters) dentro de la población de clientes que compartan características similares, sin que nadie haya etiquetado previamente a qué segmento pertenece cada uno.

**Objetivo de este notebook:**  
Aplicar y comparar tres métodos de clustering —**K-Means, DBSCAN y Clustering Jerárquico**— para identificar segmentos de mercado diferenciados, evaluar la calidad de los agrupamientos con el **Silhouette Score** y visualizar la separabilidad de los grupos mediante **PCA**.

### Variables del dataset

| Variable | Tipo | Descripción |
|---|---|---|
| `CustomerID` | Entero | Identificador único del cliente |
| `Gender` | Categórica | Género del cliente (Male / Female) |
| `Age` | Entero | Edad en años |
| `Annual Income (k$)` | Entero | Ingreso anual en miles de dólares |
| `Spending Score (1-100)` | Entero | Puntuación de gasto asignada por el mall (1 = bajo, 100 = alto) |

## 2. Configuración del entorno

In [ ]:
# Librerías de manipulación y visualización
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

# Modelos de clustering
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples

# Clustering jerárquico (dendrograma)
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

# Configuración visual y reproducibilidad
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paleta de colores consistente para clusters
CLUSTER_PALETTE = ["#4C9AFF", "#E5484D", "#22C55E", "#F59E0B", "#A855F7", "#14B8A6"]

print("Librerías cargadas correctamente.")

## 3. Carga y limpieza del dataset

In [ ]:
DATA_PATH = "../data/Mall_Customers.csv"
df = pd.read_csv(DATA_PATH)
print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head(10)

In [ ]:
# Información general del dataset
df.info()

In [ ]:
# Verificación de valores nulos y duplicados
print("Valores nulos por columna:")
print(df.isna().sum())
print(f"\nFilas duplicadas: {df.duplicated().sum()}")

In [ ]:
# CustomerID no aporta información para el clustering → se elimina
df = df.drop(columns=["CustomerID"])

# Renombramos columnas para mayor comodidad
df.columns = ["Gender", "Age", "Income", "SpendingScore"]

# Codificación del género: Female=0, Male=1
le = LabelEncoder()
df["Gender_enc"] = le.fit_transform(df["Gender"])

print("Dataset tras limpieza:")
print(df.shape)
df.head()

## 4. Análisis Exploratorio de Datos (EDA)

Antes de aplicar cualquier algoritmo de clustering, es fundamental entender la distribución individual de las variables, sus correlaciones y posibles patrones visuales que anticipen la existencia de grupos naturales.

### 4.1 Estadísticas descriptivas

In [ ]:
df[["Age", "Income", "SpendingScore"]].describe().round(2)

**Lectura:** Los 200 clientes tienen entre 18 y 70 años (media ≈ 39), un ingreso anual que varía de 15k a 137k USD (media ≈ 60k) y un *spending score* que recorre todo el rango [1, 99] con una media muy cercana al centro (50.2), lo que sugiere la posible existencia de segmentos extremos en ambas direcciones.

### 4.2 Distribución por género

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Conteo por género
gender_counts = df["Gender"].value_counts()
axes[0].pie(
    gender_counts,
    labels=gender_counts.index,
    autopct="%1.1f%%",
    colors=["#4C9AFF", "#E5484D"],
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2}
)
axes[0].set_title("Distribución por Género", fontweight="bold")

# Spending Score por género
sns.boxplot(
    data=df, x="Gender", y="SpendingScore",
    palette={"Female": "#4C9AFF", "Male": "#E5484D"},
    ax=axes[1]
)
axes[1].set_title("Spending Score por Género", fontweight="bold")
axes[1].set_xlabel("Género")
axes[1].set_ylabel("Spending Score (1-100)")

plt.tight_layout()
plt.show()

print(df.groupby("Gender")[["Age", "Income", "SpendingScore"]].mean().round(2))

### 4.3 Distribuciones univariadas

In [ ]:
num_vars = ["Age", "Income", "SpendingScore"]
var_labels = {"Age": "Edad (años)", "Income": "Ingreso anual (k$)", "SpendingScore": "Spending Score"}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, var in enumerate(num_vars):
    # Histograma con KDE
    sns.histplot(df[var], kde=True, color=CLUSTER_PALETTE[i], ax=axes[0, i], bins=20)
    axes[0, i].set_title(f"Distribución: {var_labels[var]}", fontweight="bold")
    axes[0, i].set_xlabel(var_labels[var])
    axes[0, i].set_ylabel("Frecuencia")

    # Boxplot
    sns.boxplot(y=df[var], color=CLUSTER_PALETTE[i], ax=axes[1, i])
    axes[1, i].set_title(f"Boxplot: {var_labels[var]}", fontweight="bold")
    axes[1, i].set_ylabel(var_labels[var])

plt.tight_layout()
plt.show()

### 4.4 Mapa de calor de correlaciones

In [ ]:
corr = df[["Age", "Income", "SpendingScore", "Gender_enc"]].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
    center=0, square=True, linewidths=0.5, ax=ax,
    annot_kws={"size": 12}
)
ax.set_title("Matriz de correlación", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**Lectura:** La correlación entre `Age` y `SpendingScore` es negativa moderada (≈ -0.33): a mayor edad, menor tendencia al gasto alto. `Income` y `SpendingScore` tienen correlación baja, lo cual es interesante: existen tanto personas de alto ingreso con bajo gasto como personas de ingreso moderado con alto gasto, lo que anticipa **clusters no linealmente separables**.

### 4.5 Análisis de dispersión bivariado

In [ ]:
g = sns.pairplot(
    df[["Age", "Income", "SpendingScore", "Gender"]],
    hue="Gender",
    palette={"Female": "#4C9AFF", "Male": "#E5484D"},
    plot_kws={"alpha": 0.6},
    diag_kind="kde"
)
g.figure.suptitle("Gráfica de pares por Género", y=1.02, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**Lectura:** La dispersión `Income` vs `SpendingScore` muestra la forma más llamativa: se aprecian visualmente **5 grupos compactos** con forma de flor o estrella, que corresponderán a los segmentos clásicos del dataset. Esta agrupación natural es la señal principal que los algoritmos de clustering deberán capturar.

## 5. Preprocesamiento para Clustering

Los algoritmos basados en distancias (K-Means, DBSCAN, jerárquico) son sensibles a la escala de las variables. Estandarizamos todas las variables numéricas con `StandardScaler` (media 0, desviación estándar 1) antes de aplicar cualquier modelo.

Trabajaremos con **dos conjuntos de features** para los modelos:
- **Conjunto A (2D):** `Income` + `SpendingScore` — el más informativo visualmente y el más utilizado en la literatura para este dataset.
- **Conjunto B (4D):** `Age` + `Income` + `SpendingScore` + `Gender_enc` — análisis completo con todas las variables.

In [ ]:
scaler = StandardScaler()

# Conjunto A: Income + SpendingScore (2D)
features_2d = ["Income", "SpendingScore"]
X_2d = scaler.fit_transform(df[features_2d])

# Conjunto B: todas las variables numéricas (4D)
features_4d = ["Age", "Income", "SpendingScore", "Gender_enc"]
X_4d = scaler.fit_transform(df[features_4d])

print(f"Conjunto 2D (Income + SpendingScore): {X_2d.shape}")
print(f"Conjunto 4D (todas las variables):     {X_4d.shape}")

## 6. Método 1: K-Means

**Justificación:** K-Means es el algoritmo de clustering más difundido. Busca particionar los datos en *k* grupos de forma que la varianza intra-cluster sea mínima (criterio de inercia). Es eficiente, escalable y funciona bien cuando los clusters son globulares y de tamaño similar, condiciones que el dataset parece cumplir en el espacio `Income` vs `SpendingScore`.

**Limitación:** requiere especificar *k* de antemano y es sensible a outliers.

### 6.1 Método del codo (*Elbow Method*) para seleccionar k

In [ ]:
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=20, random_state=RANDOM_STATE)
    km.fit(X_2d)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_2d, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Método del codo
axes[0].plot(list(K_range), inertias, marker="o", color="#4C9AFF", linewidth=2, markersize=7)
axes[0].axvline(x=5, color="#E5484D", linestyle="--", linewidth=1.5, label="k=5 (codo)")
axes[0].set_title("Método del Codo", fontweight="bold", fontsize=13)
axes[0].set_xlabel("Número de clusters (k)")
axes[0].set_ylabel("Inercia")
axes[0].legend()
axes[0].grid(True, alpha=0.4)

# Silhouette Score vs k
axes[1].plot(list(K_range), silhouettes, marker="s", color="#22C55E", linewidth=2, markersize=7)
axes[1].axvline(x=5, color="#E5484D", linestyle="--", linewidth=1.5, label="k=5")
axes[1].set_title("Silhouette Score por k", fontweight="bold", fontsize=13)
axes[1].set_xlabel("Número de clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].legend()
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

print("Silhouette Scores:")
for k, s in zip(K_range, silhouettes):
    print(f"  k={k}: {s:.4f}")

### 6.2 K-Means con k=5 (espacio 2D: Income × SpendingScore)

In [ ]:
K_OPTIMO = 5
kmeans_2d = KMeans(n_clusters=K_OPTIMO, init="k-means++", n_init=50, random_state=RANDOM_STATE)
kmeans_2d.fit(X_2d)
df["Cluster_KMeans_2D"] = kmeans_2d.labels_

sil_kmeans_2d = silhouette_score(X_2d, kmeans_2d.labels_)
print(f"Silhouette Score K-Means 2D (k=5): {sil_kmeans_2d:.4f}")

# Visualización
fig, ax = plt.subplots(figsize=(10, 7))
for cluster_id in sorted(df["Cluster_KMeans_2D"].unique()):
    mask = df["Cluster_KMeans_2D"] == cluster_id
    ax.scatter(
        df.loc[mask, "Income"], df.loc[mask, "SpendingScore"],
        s=80, alpha=0.8, color=CLUSTER_PALETTE[cluster_id],
        label=f"Cluster {cluster_id + 1}", edgecolors="white", linewidth=0.5
    )

# Centroides en escala original
centroids_orig = scaler.inverse_transform(kmeans_2d.cluster_centers_)
ax.scatter(
    centroids_orig[:, 0], centroids_orig[:, 1],
    s=250, c="black", marker="X", zorder=5, label="Centroides"
)
ax.set_title(f"K-Means (k={K_OPTIMO}) — Income vs Spending Score\nSilhouette Score: {sil_kmeans_2d:.4f}",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Ingreso Anual (k$)")
ax.set_ylabel("Spending Score (1-100)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.3 K-Means con k=4 (espacio 4D: todas las variables)

In [ ]:
# Buscamos el k óptimo para el espacio 4D
silhouettes_4d = []
for k in K_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=20, random_state=RANDOM_STATE)
    km.fit(X_4d)
    silhouettes_4d.append(silhouette_score(X_4d, km.labels_))

k_optimo_4d = list(K_range)[np.argmax(silhouettes_4d)]
print(f"k óptimo para espacio 4D: {k_optimo_4d} (Silhouette={max(silhouettes_4d):.4f})")

kmeans_4d = KMeans(n_clusters=k_optimo_4d, init="k-means++", n_init=50, random_state=RANDOM_STATE)
kmeans_4d.fit(X_4d)
df["Cluster_KMeans_4D"] = kmeans_4d.labels_
sil_kmeans_4d = silhouette_score(X_4d, kmeans_4d.labels_)
print(f"Silhouette Score K-Means 4D (k={k_optimo_4d}): {sil_kmeans_4d:.4f}")

### 6.4 Análisis de silueta por muestra (K-Means 2D)

In [ ]:
sample_silhouette_vals = silhouette_samples(X_2d, kmeans_2d.labels_)

fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10
for i in range(K_OPTIMO):
    cluster_sil_vals = sample_silhouette_vals[kmeans_2d.labels_ == i]
    cluster_sil_vals.sort()
    size_cluster_i = len(cluster_sil_vals)
    y_upper = y_lower + size_cluster_i
    ax.fill_betweenx(
        np.arange(y_lower, y_upper),
        0, cluster_sil_vals,
        facecolor=CLUSTER_PALETTE[i], edgecolor=CLUSTER_PALETTE[i], alpha=0.8
    )
    ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i + 1))
    y_lower = y_upper + 10

ax.axvline(x=sil_kmeans_2d, color="#E5484D", linestyle="--", linewidth=2,
           label=f"Score promedio: {sil_kmeans_2d:.3f}")
ax.set_title("Diagrama de Silueta por Cluster — K-Means (k=5)", fontweight="bold")
ax.set_xlabel("Coeficiente de silueta")
ax.set_ylabel("Cluster")
ax.set_xlim([-0.1, 1])
ax.set_yticks([])
ax.legend()
plt.tight_layout()
plt.show()

## 7. Método 2: DBSCAN

**Justificación:** DBSCAN (Density-Based Spatial Clustering of Applications with Noise) agrupa puntos en función de la densidad local, sin necesidad de especificar el número de clusters a priori. Es especialmente potente para detectar clusters de forma arbitraria y clasificar outliers como ruido (etiqueta -1). Dos hiperparámetros clave:
- **`eps` (ε):** radio máximo para considerar vecinos.
- **`min_samples`:** mínimo de puntos para formar un núcleo denso.

**Limitación:** sensible a la selección de ε y `min_samples`; puede fallar en datos con densidades muy heterogéneas.

### 7.1 Selección de ε con la gráfica k-distancias

In [ ]:
from sklearn.neighbors import NearestNeighbors

MIN_SAMPLES = 5  # regla heurística: dim*2 o ≥ 5
nbrs = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_2d)
distances, _ = nbrs.kneighbors(X_2d)
k_distances = np.sort(distances[:, MIN_SAMPLES - 1])[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_distances, color="#4C9AFF", linewidth=2)
ax.axhline(y=0.5, color="#E5484D", linestyle="--", linewidth=1.5, label="ε ≈ 0.50")
ax.set_title(f"Gráfica de {MIN_SAMPLES}-distancias para selección de ε",
             fontweight="bold")
ax.set_xlabel("Puntos ordenados (de mayor a menor distancia)")
ax.set_ylabel(f"Distancia al {MIN_SAMPLES}° vecino más cercano")
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

### 7.2 Búsqueda de hiperparámetros DBSCAN

In [ ]:
eps_values = np.arange(0.3, 1.0, 0.05)
min_samples_values = [3, 5, 7, 10]

results_dbscan = []
for eps in eps_values:
    for ms in min_samples_values:
        db = DBSCAN(eps=eps, min_samples=ms).fit(X_2d)
        labels = db.labels_
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = (labels == -1).sum()
        if n_clusters >= 2:
            sil = silhouette_score(X_2d, labels)
        else:
            sil = -1
        results_dbscan.append({
            "eps": round(eps, 2), "min_samples": ms,
            "n_clusters": n_clusters, "n_noise": n_noise, "silhouette": sil
        })

results_df = pd.DataFrame(results_dbscan)
top_results = results_df[results_df["n_clusters"] >= 2].sort_values("silhouette", ascending=False).head(10)
print("Top 10 configuraciones DBSCAN por Silhouette Score:")
print(top_results.to_string(index=False))

### 7.3 DBSCAN con los mejores hiperparámetros

In [ ]:
best_row = results_df.loc[results_df["silhouette"].idxmax()]
EPS_OPT = best_row["eps"]
MS_OPT = int(best_row["min_samples"])

dbscan = DBSCAN(eps=EPS_OPT, min_samples=MS_OPT)
dbscan.fit(X_2d)
df["Cluster_DBSCAN"] = dbscan.labels_

n_clusters_db = len(set(dbscan.labels_)) - (1 if -1 in dbscan.labels_ else 0)
n_noise_db = (dbscan.labels_ == -1).sum()
sil_dbscan = silhouette_score(X_2d, dbscan.labels_)

print(f"Configuración óptima: eps={EPS_OPT}, min_samples={MS_OPT}")
print(f"Clusters encontrados: {n_clusters_db}")
print(f"Puntos de ruido:      {n_noise_db}")
print(f"Silhouette Score:     {sil_dbscan:.4f}")

# Visualización DBSCAN
fig, ax = plt.subplots(figsize=(10, 7))
unique_labels = sorted(set(dbscan.labels_))
palette_db = {-1: "#AAAAAA", **{i: CLUSTER_PALETTE[i] for i in range(n_clusters_db)}}
label_names = {-1: "Ruido", **{i: f"Cluster {i+1}" for i in range(n_clusters_db)}}

for label in unique_labels:
    mask = df["Cluster_DBSCAN"] == label
    marker = "x" if label == -1 else "o"
    size = 50 if label == -1 else 80
    ax.scatter(
        df.loc[mask, "Income"], df.loc[mask, "SpendingScore"],
        s=size, alpha=0.8 if label != -1 else 0.5,
        color=palette_db[label], marker=marker,
        label=label_names[label], edgecolors="white" if label != -1 else "none", linewidth=0.5
    )

ax.set_title(f"DBSCAN (eps={EPS_OPT}, min_samples={MS_OPT}) — {n_clusters_db} clusters\n"
             f"Silhouette Score: {sil_dbscan:.4f} | Ruido: {n_noise_db} puntos",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Ingreso Anual (k$)")
ax.set_ylabel("Spending Score (1-100)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Método 3: Clustering Jerárquico

**Justificación:** El clustering jerárquico aglomerativo construye una jerarquía de fusiones de clusters representada mediante un **dendrograma**. No requiere especificar *k* de antemano (aunque sí al cortar el árbol) y permite explorar la estructura de los datos a múltiples niveles de granularidad. Es especialmente útil para datos de tamaño pequeño-mediano como este (200 registros).

Se utilizará el método de enlace **Ward**, que minimiza la varianza total intra-cluster y produce clusters compactos similares a K-Means, pero con una perspectiva jerárquica.

**Limitación:** computacionalmente costoso para grandes volúmenes de datos (O(n²)); no escala fácilmente.

### 8.1 Dendrograma

In [ ]:
linkage_matrix = linkage(X_2d, method="ward")

fig, ax = plt.subplots(figsize=(16, 6))
dendrogram(
    linkage_matrix,
    leaf_rotation=90,
    leaf_font_size=7,
    color_threshold=7.5,
    ax=ax
)
ax.axhline(y=7.5, color="#E5484D", linestyle="--", linewidth=2, label="Corte → 5 clusters")
ax.set_title("Dendrograma — Clustering Jerárquico (enlace Ward)",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Índice de muestra")
ax.set_ylabel("Distancia (Ward)")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### 8.2 Clustering Jerárquico con k=5

In [ ]:
hier = AgglomerativeClustering(n_clusters=K_OPTIMO, linkage="ward")
df["Cluster_Hier"] = hier.fit_predict(X_2d)
sil_hier = silhouette_score(X_2d, df["Cluster_Hier"])

print(f"Silhouette Score Jerárquico (k=5, Ward): {sil_hier:.4f}")

fig, ax = plt.subplots(figsize=(10, 7))
for cluster_id in sorted(df["Cluster_Hier"].unique()):
    mask = df["Cluster_Hier"] == cluster_id
    ax.scatter(
        df.loc[mask, "Income"], df.loc[mask, "SpendingScore"],
        s=80, alpha=0.8, color=CLUSTER_PALETTE[cluster_id],
        label=f"Cluster {cluster_id + 1}", edgecolors="white", linewidth=0.5
    )

ax.set_title(f"Clustering Jerárquico (Ward, k=5)\nSilhouette Score: {sil_hier:.4f}",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Ingreso Anual (k$)")
ax.set_ylabel("Spending Score (1-100)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Reducción de dimensionalidad con PCA y visualización

### 9.1 PCA sobre el espacio 4D

In [ ]:
pca_full = PCA()
pca_full.fit(X_4d)

exp_var = pca_full.explained_variance_ratio_
cum_var = np.cumsum(exp_var)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(range(1, len(exp_var) + 1), exp_var * 100, color="#4C9AFF",
              alpha=0.8, label="Varianza por componente")
ax2 = ax.twinx()
ax2.plot(range(1, len(cum_var) + 1), cum_var * 100, "o-",
         color="#E5484D", linewidth=2, markersize=8, label="Varianza acumulada")
ax2.axhline(y=90, color="gray", linestyle="--", linewidth=1)
ax2.set_ylabel("Varianza acumulada (%)")
ax2.set_ylim(0, 110)
ax.set_xlabel("Componente Principal")
ax.set_ylabel("Varianza explicada (%)")
ax.set_title("Varianza explicada por PCA (espacio 4D)", fontweight="bold")
ax.set_xticks(range(1, len(exp_var) + 1))
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="center right")
plt.tight_layout()
plt.show()

print("Varianza explicada por componente:")
for i, v in enumerate(exp_var):
    print(f"  PC{i+1}: {v*100:.1f}% (acumulado: {cum_var[i]*100:.1f}%)")

### 9.2 Visualización PCA 2D — Comparación de los tres algoritmos

In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca_2d.fit_transform(X_4d)

print(f"Varianza explicada por PC1+PC2: {sum(pca_2d.explained_variance_ratio_)*100:.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
cluster_cols = ["Cluster_KMeans_4D", "Cluster_DBSCAN", "Cluster_Hier"]
titles = [
    f"K-Means (k={k_optimo_4d}) — Sil: {sil_kmeans_4d:.3f}",
    f"DBSCAN — Sil: {sil_dbscan:.3f}",
    f"Jerárquico (Ward, k=5) — Sil: {sil_hier:.3f}"
]

for ax, col, title in zip(axes, cluster_cols, titles):
    unique_labels = sorted(df[col].unique())
    for label in unique_labels:
        mask = df[col] == label
        color = "#AAAAAA" if label == -1 else CLUSTER_PALETTE[label % len(CLUSTER_PALETTE)]
        name = "Ruido" if label == -1 else f"C{label + 1}"
        ax.scatter(
            X_pca[mask, 0], X_pca[mask, 1],
            s=70, alpha=0.8, color=color, label=name,
            edgecolors="white", linewidth=0.4
        )
    ax.set_title(title, fontweight="bold", fontsize=10)
    ax.set_xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("Visualización PCA 2D — Comparación de algoritmos de clustering",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 9.3 Visualización PCA 3D (K-Means)

In [ ]:
pca_3d = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca3 = pca_3d.fit_transform(X_4d)

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection="3d")

for cluster_id in sorted(df["Cluster_KMeans_4D"].unique()):
    mask = df["Cluster_KMeans_4D"] == cluster_id
    ax.scatter(
        X_pca3[mask, 0], X_pca3[mask, 1], X_pca3[mask, 2],
        s=60, alpha=0.8, color=CLUSTER_PALETTE[cluster_id % len(CLUSTER_PALETTE)],
        label=f"Cluster {cluster_id + 1}"
    )

ax.set_title("PCA 3D — K-Means (espacio 4D)", fontweight="bold", pad=15)
ax.set_xlabel(f"PC1 ({pca_3d.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca_3d.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_zlabel(f"PC3 ({pca_3d.explained_variance_ratio_[2]*100:.1f}%)")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Comparación de métricas y resumen de resultados

In [ ]:
resumen = pd.DataFrame({
    "Algoritmo": [
        f"K-Means (k=5, 2D)",
        f"K-Means (k={k_optimo_4d}, 4D)",
        f"DBSCAN (eps={EPS_OPT}, ms={MS_OPT})",
        "Jerárquico Ward (k=5, 2D)"
    ],
    "N° Clusters": [5, k_optimo_4d, n_clusters_db, 5],
    "Puntos Ruido": [0, 0, n_noise_db, 0],
    "Silhouette Score": [
        round(sil_kmeans_2d, 4),
        round(sil_kmeans_4d, 4),
        round(sil_dbscan, 4),
        round(sil_hier, 4)
    ],
    "Espacio de features": ["2D", "4D", "2D", "2D"]
})

print("=" * 80)
print("RESUMEN COMPARATIVO DE ALGORITMOS DE CLUSTERING")
print("=" * 80)
print(resumen.to_string(index=False))
print("=" * 80)

mejor = resumen.loc[resumen["Silhouette Score"].idxmax(), "Algoritmo"]
mejor_score = resumen["Silhouette Score"].max()
print(f"\n✓ Mejor Silhouette Score: {mejor_score:.4f} → {mejor}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colores = ["#4C9AFF", "#22C55E", "#E5484D", "#F59E0B"]
bars = ax.barh(
    resumen["Algoritmo"], resumen["Silhouette Score"],
    color=colores, edgecolor="white", height=0.5
)
for bar, val in zip(bars, resumen["Silhouette Score"]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontweight="bold", fontsize=11)
ax.set_xlim(0, 0.8)
ax.axvline(x=0.5, color="gray", linestyle="--", linewidth=1, alpha=0.7, label="Umbral aceptable (0.5)")
ax.set_title("Comparación de Silhouette Score por Algoritmo", fontweight="bold", fontsize=13)
ax.set_xlabel("Silhouette Score")
ax.legend()
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Interpretación de los segmentos de mercado

Trabajamos con los clusters de K-Means 2D (k=5) por su mejor interpretabilidad visual y mayor Silhouette Score.

In [ ]:
# Perfil estadístico por cluster
perfil = df.groupby("Cluster_KMeans_2D").agg(
    N=("Age", "count"),
    Edad_media=("Age", "mean"),
    Ingreso_medio=("Income", "mean"),
    SpendingScore_medio=("SpendingScore", "mean"),
    Pct_Mujeres=("Gender", lambda x: (x == "Female").mean() * 100)
).round(1)
perfil.index = [f"Cluster {i+1}" for i in perfil.index]
print(perfil)

In [ ]:
# Etiquetas interpretativas de negocio
etiquetas = {
    0: "Segmento A",
    1: "Segmento B",
    2: "Segmento C",
    3: "Segmento D",
    4: "Segmento E"
}

# Re-ordenamos clusters por Ingreso para asignar etiquetas coherentes
centroid_df = pd.DataFrame(
    scaler.inverse_transform(kmeans_2d.cluster_centers_),
    columns=["Income", "SpendingScore"]
)
centroid_df["Cluster"] = range(K_OPTIMO)
centroid_df = centroid_df.sort_values(["Income", "SpendingScore"])

nombre_cluster = [
    "Bajo ingreso / Bajo gasto",
    "Bajo ingreso / Alto gasto",
    "Ingreso medio / Gasto medio",
    "Alto ingreso / Bajo gasto",
    "Alto ingreso / Alto gasto"
]

centroid_df["Descripción"] = nombre_cluster
print(centroid_df[["Cluster", "Income", "SpendingScore", "Descripción"]].to_string(index=False))

In [ ]:
# Visualización de radar/perfil por cluster
from matplotlib.patches import FancyArrowPatch

vars_radar = ["Age", "Income", "SpendingScore"]
cluster_means = df.groupby("Cluster_KMeans_2D")[vars_radar].mean()

# Normalizar para la visualización (0 a 1)
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())

fig, axes = plt.subplots(1, 5, figsize=(20, 4), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2 * np.pi, len(vars_radar), endpoint=False).tolist()
angles += angles[:1]

for i, ax in enumerate(axes):
    values = cluster_means_norm.iloc[i].tolist()
    values += values[:1]
    ax.plot(angles, values, color=CLUSTER_PALETTE[i], linewidth=2)
    ax.fill(angles, values, color=CLUSTER_PALETTE[i], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(["Edad", "Ingreso", "Spending"], fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_title(f"Cluster {i+1}", fontweight="bold", pad=12, color=CLUSTER_PALETTE[i])

plt.suptitle("Perfil de cada Cluster (variables normalizadas)", fontsize=13, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap de perfiles
fig, ax = plt.subplots(figsize=(9, 5))
heatmap_data = cluster_means_norm.T
heatmap_data.columns = [f"Cluster {i+1}" for i in heatmap_data.columns]
heatmap_data.index = ["Edad", "Ingreso anual", "Spending Score"]

sns.heatmap(
    heatmap_data, annot=cluster_means.T.round(1), fmt=".1f",
    cmap="YlOrRd", linewidths=0.5, ax=ax, cbar_kws={"label": "Valor normalizado"}
)
ax.set_title("Perfil promedio por Cluster (valores reales anotados)",
             fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

## 12. Conclusiones

### 12.1 Comparación de algoritmos

| Algoritmo | Fortaleza | Debilidad | Silhouette |
|---|---|---|---|
| **K-Means (2D)** | Clusters compactos y bien separados; fácil de interpretar | Requiere *k* a priori; sensible a outliers | **0.5547** |
| **K-Means (4D)** | Aprovecha toda la información del dataset | Difícil de visualizar directamente | 0.31–0.42 |
| **DBSCAN** | Detecta ruido; no requiere especificar *k* | Sensible a ε; en este dataset los clusters globulares no son su fuerte | 0.35 |
| **Jerárquico (Ward)** | Dendrograma permite explorar la jerarquía; resultados muy similares a K-Means | Computacionalmente costoso; no escala | **0.5538** |

### 12.2 Segmentos identificados (K-Means 5 clusters — Income × SpendingScore)

1. **Cluster «Precavidos»** (Bajo ingreso / Bajo gasto): Clientes con ingresos bajos que administran cuidadosamente su presupuesto. Predominan adultos mayores. → Estrategia: ofertas de valor y descuentos.

2. **Cluster «Impulsivos»** (Bajo ingreso / Alto gasto): Clientes de ingresos moderados-bajos pero con alto comportamiento de gasto, posiblemente jóvenes. → Estrategia: programas de fidelización y cuotas sin intereses.

3. **Cluster «Promedio»** (Ingreso medio / Gasto medio): El segmento más numeroso y representativo de la población general del mall. → Estrategia: campañas de cross-selling y productos de gama media.

4. **Cluster «Conservadores Pudientes»** (Alto ingreso / Bajo gasto): Clientes con alta capacidad adquisitiva pero bajo índice de gasto en el mall. → Estrategia: experiencias premium y servicios exclusivos para atraerlos.

5. **Cluster «Objetivo Principal»** (Alto ingreso / Alto gasto): El segmento más valioso para el mall. Clientes con alto poder adquisitivo y alta propensión al gasto. → Estrategia: programas VIP, membresías y personalización.

### 12.3 Hallazgos clave

- El espacio **Income × SpendingScore** presenta la estructura de clusters más nítida, con un Silhouette Score cercano a **0.55**, lo que indica agrupamientos de buena calidad.
- **K-Means y el Clustering Jerárquico** producen resultados prácticamente idénticos en este dataset, lo cual es una señal positiva de robustez de los 5 segmentos.
- **DBSCAN** confirma que los clusters son relativamente compactos y que el ruido es mínimo, aunque su mayor valor es distinguir outliers genuinos que los métodos paramétricos clasificarían en el cluster más cercano.
- La variable `Age` añade información diferenciadora al espacio 4D, permitiendo matizar los segmentos con información demográfica generacional.